In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração mais leve para o Colab
plt.rcParams["font.family"] = "DejaVu Sans"
sns.set_theme(style="whitegrid")

In [ ]:
# Identifica automaticamente o nome real do arquivo enviado
nome_arquivo = next(iter(uploaded))

df = pd.read_csv(io.BytesIO(uploaded[nome_arquivo]))

print(f'Arquivo carregado: {nome_arquivo}')
print(f'Dimensões da base: {df.shape}')
display(df.head())


In [ ]:
# ============================================================
# EDA — Base Roney Vinicius
# Versão corrigida para Google Colab
# ============================================================

import os
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
from IPython.display import display

# 1. Upload do CSV
uploaded = files.upload()

if not uploaded:
    raise FileNotFoundError('Nenhum arquivo foi enviado.')

nome_arquivo = next(iter(uploaded))
print(f'Arquivo selecionado: {nome_arquivo}')

# O arquivo enviado pelo Colab fica disponível no diretório atual (/content)
INPUT = nome_arquivo
df = pd.read_csv(INPUT)

# 2. Pasta de saída do Colab
OUT = '/content/eda_output'
os.makedirs(OUT, exist_ok=True)

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['font.family'] = 'DejaVu Sans'

# Paleta inspirada no notebook de referência.
PALETTE = {'fake': '#de425b', 'true': '#488f31'}

print(f'Dimensões da base: {df.shape[0]:,} linhas x {df.shape[1]} colunas')
display(df.head())

# 3. Conversão robusta das datas
# A base mistura formatos como:
#   2017-11-30
#   2022-09-08T00:00:00Z
#   01/02/2018
#   11/1/2018
# As datas com barras são interpretadas como dia/mês/ano.
if 'date_of_publication' in df.columns:
    datas = df['date_of_publication'].astype('string').str.strip()

    df['date_parsed'] = pd.Series(
        pd.NaT,
        index=df.index,
        dtype='datetime64[ns, UTC]'
    )

    mask_iso = datas.str.match(r'^\d{4}-\d{2}-\d{2}', na=False)
    mask_br = datas.str.match(r'^\d{1,2}/\d{1,2}/\d{4}', na=False)

    df.loc[mask_iso, 'date_parsed'] = pd.to_datetime(
        datas[mask_iso],
        format='ISO8601',
        errors='coerce',
        utc=True
    )

    df.loc[mask_br, 'date_parsed'] = pd.to_datetime(
        datas[mask_br],
        dayfirst=True,
        errors='coerce',
        utc=True
    )

    print('\nDatas:')
    print(f'- Valores originais não vazios: {datas.notna().sum():,}')
    print(f'- Datas convertidas corretamente: {df["date_parsed"].notna().sum():,}')
    print(f'- Primeira data: {df["date_parsed"].min()}')
    print(f'- Última data: {df["date_parsed"].max()}')
else:
    df['date_parsed'] = pd.NaT
    print('A coluna date_of_publication não foi encontrada.')

# 4. Tipos de variáveis
numeric = df.select_dtypes(include=np.number).columns.tolist()
categorical = [c for c in df.columns if c not in numeric]

# 5. Resumo principal
valid_dates = df['date_parsed'].dropna()
summary = {
    'arquivo': nome_arquivo,
    'n_rows': int(len(df)),
    'n_columns': int(df.shape[1] - 1),  # exclui date_parsed, coluna criada pela análise
    'duplicate_rows': int(df.duplicated().sum()),
    'duplicate_links': int(df['link'].duplicated().sum()) if 'link' in df.columns else None,
    'missing_cells_original': int(df.drop(columns=['date_parsed']).isna().sum().sum()),
    'date_valid_count': int(valid_dates.size),
    'date_invalid_or_missing_count': int(len(df) - valid_dates.size),
    'date_min': str(valid_dates.min().date()) if len(valid_dates) else None,
    'date_max': str(valid_dates.max().date()) if len(valid_dates) else None,
}

with open(os.path.join(OUT, 'summary.json'), 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

# Tabelas de qualidade e estatística
original_df = df.drop(columns=['date_parsed'])
original_df.isna().sum().rename('missing').to_csv(
    os.path.join(OUT, 'missing_by_column.csv')
)

df[numeric].describe().T.to_csv(
    os.path.join(OUT, 'numeric_describe.csv')
)

if 'label' in df.columns:
    label_table = df['label'].value_counts(dropna=False).rename('count').to_frame()
    label_table['percent'] = label_table['count'] / len(df) * 100
    label_table.to_csv(
        os.path.join(OUT, 'label_counts.csv')
    )

if 'category' in df.columns:
    category_table = df['category'].value_counts(dropna=False).rename('count').to_frame()
    category_table['percent'] = category_table['count'] / len(df) * 100
    category_table.to_csv(
        os.path.join(OUT, 'category_counts.csv')
    )

# 6. Distribuição dos rótulos e categorias
fig, axes = plt.subplots(
    1, 2,
    figsize=(19, 7),
    gridspec_kw={'width_ratios': [1, 1.7]}
)

if 'label' in df.columns:
    vc = df['label'].value_counts(normalize=True).mul(100)
    colors = [PALETTE.get(str(x).lower(), '#4472C4') for x in vc.index]
    sns.barplot(x=vc.index, y=vc.values, ax=axes[0], palette=colors, hue=vc.index, legend=False)
    axes[0].set_title('Distribuição Percentual dos Rótulos', fontsize=14, fontweight='bold', pad=15)
    axes[0].set_xlabel('Tipo de Notícia', fontsize=12)
    axes[0].set_ylabel('Percentual (%)', fontsize=12)
    for i, v in enumerate(vc.values):
        axes[0].text(i, v, f'{v:.1f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')
else:
    axes[0].axis('off')

if 'category' in df.columns and 'label' in df.columns:
    # Percentual de fake/true dentro de cada categoria.
    # Cada categoria soma 100%, facilitando a comparação entre grupos.
    category_label_pct = pd.crosstab(
        df['category'], df['label'], normalize='index'
    ).mul(100)
    label_order = [x for x in ['fake', 'true'] if x in category_label_pct.columns]
    category_label_pct = category_label_pct[label_order]
    category_label_pct = category_label_pct.loc[
        category_label_pct.sum(axis=1).sort_values(ascending=True).index
    ]

    category_label_pct.plot(
        kind='barh',
        ax=axes[1],
        color=[PALETTE.get(str(x).lower(), '#4472C4') for x in label_order],
        width=0.8
    )
    axes[1].set_title(
        'Distribuição Percentual por Categoria e Grupo',
        fontsize=14, fontweight='bold', pad=15
    )
    axes[1].set_xlabel('Percentual dentro da categoria (%)', fontsize=12)
    axes[1].set_ylabel('Categoria da Notícia', fontsize=12)
    axes[1].set_yticks(range(len(category_label_pct.index)))
    axes[1].set_yticklabels(category_label_pct.index, rotation=0, fontsize=10)
    axes[1].tick_params(axis='y', labelleft=True, pad=8)
    axes[1].legend(title='Grupo', fontsize=10, title_fontsize=11)
    for container in axes[1].containers:
        axes[1].bar_label(
            container, fmt='%.1f%%', padding=3, fontsize=9,
            fontweight='bold', label_type='center'
        )
elif 'category' in df.columns:
    vc2 = df['category'].value_counts(normalize=True).mul(100).sort_values(ascending=True)
    sns.barplot(x=vc2.values, y=vc2.index, ax=axes[1], color='#488f31')
    axes[1].set_title('Distribuição Percentual por Categoria', fontsize=14, fontweight='bold', pad=15)
    axes[1].set_xlabel('Percentual (%)', fontsize=12)
    axes[1].set_ylabel('Categoria', fontsize=12)
    for container in axes[1].containers:
        axes[1].bar_label(container, fmt='%.1f%%', padding=5, fontsize=10, fontweight='bold')

for ax in axes:
    sns.despine(ax=ax)
else:
    axes[1].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUT, '01_distribuicoes.png'), dpi=180, bbox_inches='tight')
plt.show()
plt.close()

# 7. Variáveis numéricas por rótulo
key = [c for c in [
    'number_of_tokens',
    'number_of_words_without_punctuation',
    'number_of_types',
    'number_of_links_inside_news',
    'average_sentence_length',
    'average_word_length',
    'emotiveness',
    'diversity',
    'pausality'
] if c in df.columns]

if 'label' in df.columns and key:
    ncols = 3
    nrows = int(np.ceil(len(key) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows))
    axes = np.array(axes).reshape(-1)

    for ax, c in zip(axes, key):
        sns.boxplot(
            data=df,
            x='label',
            y=c,
            ax=ax,
            showfliers=False,
            hue='label',
            palette=PALETTE,
            legend=False
        )
        ax.set_title(c.replace('_', ' '))
        ax.set_xlabel('Rótulo')
        ax.set_ylabel('')

    for ax in axes[len(key):]:
        ax.axis('off')

    plt.tight_layout()
    plt.savefig(os.path.join(OUT, '02_variaveis_por_rotulo.png'), dpi=180, bbox_inches='tight')
    plt.show()
    plt.close()

# 8. Matriz de correlação
if numeric:
    corr = df[numeric].corr(numeric_only=True)
    corr.to_csv(os.path.join(OUT, 'correlation_matrix.csv'))

    fig, ax = plt.subplots(figsize=(14, 11))
    sns.heatmap(corr, cmap='vlag', center=0, ax=ax, xticklabels=True, yticklabels=True)
    ax.set_title('Matriz de correlação das variáveis numéricas')
    plt.xticks(rotation=70, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT, '03_correlacoes.png'), dpi=180, bbox_inches='tight')
    plt.show()
    plt.close()

# 9. Evolução temporal por mês e por ano
if df['date_parsed'].notna().any():
    temp = df.dropna(subset=['date_parsed']).copy()
    temp['month'] = temp['date_parsed'].dt.tz_convert(None).dt.to_period('M').astype(str)
    temp['year'] = temp['date_parsed'].dt.year

    if 'label' in temp.columns:
        counts = temp.groupby(['month', 'label']).size().unstack(fill_value=0)
    else:
        counts = temp.groupby('month').size().to_frame('count')

    # Percentual dentro de cada mês, para comparação entre rótulos.
    counts = counts.div(counts.sum(axis=1), axis=0).mul(100)

    fig, ax = plt.subplots(figsize=(15, 5))
    counts.plot(ax=ax, marker='o')
    ax.set_title('Evolução Percentual das Notícias ao Longo do Tempo', fontsize=18, fontweight='bold', pad=20)
    ax.set_xlabel('Mês de Publicação', fontsize=14, fontweight='bold')
    ax.set_ylabel('Percentual dentro do mês (%)', fontsize=14, fontweight='bold')
    if 'label' in temp.columns:
        for line in ax.lines:
            for x, y in zip(line.get_xdata(), line.get_ydata()):
                if not pd.isna(y):
                    ax.annotate(f'{y:.1f}%', (x, y), textcoords='offset points', xytext=(0, 8),
                                ha='center', fontsize=8, fontweight='bold')
        ax.legend(title='Tipo de Notícia', title_fontsize=12, fontsize=11)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(os.path.join(OUT, '04_serie_temporal.png'), dpi=180, bbox_inches='tight')
    plt.show()
    plt.close()
    counts.to_csv(os.path.join(OUT, 'monthly_counts.csv'))

    annual_counts = temp.groupby('year').size().rename('count').to_frame()
    annual_counts['percent'] = annual_counts['count'] / len(temp) * 100
    annual_counts.to_csv(os.path.join(OUT, 'annual_counts.csv'))

    # Tabela ANO x CATEGORIA x FAKE/TRUE (%).
    # Primeiro contamos apenas as combinações que realmente existem.
    # Depois completamos rótulos ausentes com 0% e normalizamos dentro de
    # cada combinação (ano, categoria). Assim, 2012/sociedade_cotidiano,
    # que possui somente registros fake, resulta em fake=100% e true=0%.
    if 'category' in temp.columns and 'label' in temp.columns:
        valid = temp.dropna(subset=['year', 'category', 'label']).copy()
        counts_yc = pd.crosstab(
            [valid['year'], valid['category']],
            valid['label']
        )
        # Cria a grade completa: todos os anos da base x todas as categorias.
        # Combinações inexistentes recebem contagem zero, em vez de desaparecer.
        todos_os_anos = sorted(valid['year'].dropna().unique())
        todas_as_categorias = sorted(valid['category'].dropna().unique())
        grade_completa = pd.MultiIndex.from_product(
            [todos_os_anos, todas_as_categorias],
            names=['year', 'category']
        )
        counts_yc = counts_yc.reindex(grade_completa, fill_value=0)
        for grupo in ['fake', 'true']:
            if grupo not in counts_yc.columns:
                counts_yc[grupo] = 0
        counts_yc = counts_yc[['fake', 'true']]
        percent_yc = counts_yc.div(counts_yc.sum(axis=1), axis=0).mul(100).fillna(0)
        percent_yc.columns = ['fake_percent', 'true_percent']
        percent_yc['total_registros'] = counts_yc.sum(axis=1).astype(int)
        percent_yc = percent_yc.reset_index()
        percent_yc['year'] = percent_yc['year'].astype(int)
        percent_yc = percent_yc.sort_values(['year', 'category'])
        percent_yc.to_csv(
            os.path.join(OUT, 'ano_categoria_fake_true_percentual.csv'),
            index=False
        )
        print('\nTabela ANO x CATEGORIA x FAKE/TRUE (%):')
        # Exibe todas as linhas, sem head(), para não ocultar anos/categorias.
        pd.set_option('display.max_rows', None)
        display(percent_yc)

        # Gráfico legível para ANO x CATEGORIA x FAKE/TRUE (%).
        # Uma célula mostra os dois grupos; combinações sem registros ficam como "—".
        fake_heat = percent_yc.pivot(
            index='year', columns='category', values='fake_percent'
        )
        true_heat = percent_yc.pivot(
            index='year', columns='category', values='true_percent'
        )
        total_heat = percent_yc.pivot(
            index='year', columns='category', values='total_registros'
        )

        annot_heat = fake_heat.copy().astype(object)
        for ano in fake_heat.index:
            for categoria in fake_heat.columns:
                total = total_heat.loc[ano, categoria]
                if total == 0:
                    annot_heat.loc[ano, categoria] = '—'
                else:
                    annot_heat.loc[ano, categoria] = (
                        f"F {fake_heat.loc[ano, categoria]:.1f}%\n"
                        f"T {true_heat.loc[ano, categoria]:.1f}%\n"
                        f"n={int(total)}"
                    )

        plt.figure(figsize=(15, 8))
        ax_heat = sns.heatmap(
            fake_heat,
            cmap='RdYlGn_r',
            vmin=0,
            vmax=100,
            annot=annot_heat,
            fmt='',
            linewidths=0.8,
            linecolor='white',
            cbar_kws={'label': 'Percentual Fake dentro da combinação (%)'},
            annot_kws={'fontsize': 9, 'fontweight': 'bold'}
        )
        ax_heat.set_title(
            'ANO × CATEGORIA × FAKE/TRUE (%)',
            fontsize=18,
            fontweight='bold',
            pad=18
        )
        ax_heat.set_xlabel('Categoria da Notícia', fontsize=13, fontweight='bold')
        ax_heat.set_ylabel('Ano de Publicação', fontsize=13, fontweight='bold')
        ax_heat.set_xticklabels(
            ax_heat.get_xticklabels(), rotation=30, ha='right', fontsize=10
        )
        ax_heat.set_yticklabels(ax_heat.get_yticklabels(), rotation=0, fontsize=10)
        plt.tight_layout()
        plt.savefig(
            os.path.join(OUT, '07_heatmap_ano_categoria_fake_true.png'),
            dpi=220,
            bbox_inches='tight'
        )
        plt.show()
        plt.close()

    plt.figure(figsize=(10, 5))
    ax_year = sns.barplot(x=annual_counts.index.astype(str), y=annual_counts['percent'], color='#488f31')
    ax_year.bar_label(ax_year.containers[0], fmt='%.1f%%', padding=5, fontsize=10, fontweight='bold')
    plt.title('Distribuição Percentual das Notícias por Ano', fontsize=18, fontweight='bold', pad=20)
    plt.xlabel('Ano de Publicação', fontsize=14, fontweight='bold')
    plt.ylabel('Percentual do Total (%)', fontsize=14, fontweight='bold')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT, '06_distribuicao_anual.png'), dpi=180, bbox_inches='tight')
    plt.show()
    plt.close()

# 10. Número de tokens por categoria
if 'category' in df.columns and 'number_of_tokens' in df.columns:
    order = df.groupby('category')['number_of_tokens'].median().sort_values().index
    fig, ax = plt.subplots(figsize=(12, max(5, len(order) * 0.45)))
    sns.boxplot(
        data=df,
        y='category',
        x='number_of_tokens',
        order=order,
        showfliers=False,
        color='#5B9BD5',
        ax=ax
    )
    ax.set_title('Distribuição do número de tokens por categoria')
    ax.set_xlabel('Número de tokens')
    ax.set_ylabel('Categoria')
    plt.tight_layout()
    plt.savefig(os.path.join(OUT, '05_tokens_por_categoria.png'), dpi=180, bbox_inches='tight')
    plt.show()
    plt.close()

# 11. Estatísticas agrupadas
def grouped(cols, by):
    cols = [c for c in cols if c in df.columns]
    if not cols:
        return pd.DataFrame()
    return df.groupby(by)[cols].agg(['count', 'mean', 'median', 'std']).round(3)

if 'label' in df.columns:
    grouped(key, 'label').to_csv(os.path.join(OUT, 'group_stats_by_label.csv'))

if 'category' in df.columns:
    grouped([
        'number_of_tokens',
        'number_of_words_without_punctuation',
        'number_of_types',
        'average_sentence_length',
        'average_word_length',
        'emotiveness',
        'diversity'
    ], 'category').to_csv(os.path.join(OUT, 'group_stats_by_category.csv'))

# 12. Relatório Markdown
label_counts = df['label'].value_counts() if 'label' in df.columns else pd.Series(dtype=int)
cat_counts = df['category'].value_counts() if 'category' in df.columns else pd.Series(dtype=int)
miss = original_df.isna().sum().sort_values(ascending=False)
miss = miss[miss > 0]

report = []
report.append('# EDA — Base Roney Vinicius\n')
report.append(
    f"## Sumário executivo\n"
    f"A base contém **{len(df):,} registros** e **{df.shape[1] - 1} variáveis originais**. "
    f"Foram encontrados **{int(original_df.isna().sum().sum()):,} valores ausentes**, "
    f"**{int(df.duplicated().sum()):,} linhas duplicadas** e "
    f"**{int(df['link'].duplicated().sum()) if 'link' in df.columns else 0:,} links duplicados**. "
    f"Após o tratamento dos formatos mistos, o período observado vai de "
    f"**{summary['date_min']} a {summary['date_max']}**, com "
    f"**{summary['date_valid_count']:,} datas válidas**.\n"
)

if len(label_counts):
    dist = '; '.join([
        f'**{k}**: {v / len(df):.1%}'
        for k, v in label_counts.items()
    ])
    report.append(f"Os rótulos estão distribuídos assim: {dist}.\n")

report.append('## Estrutura e qualidade\n')
report.append(
    'A coluna `date_of_publication` possui formatos mistos. O tratamento separa datas ISO '
    'de datas no padrão brasileiro `dia/mês/ano`, evitando que notícias posteriores a 2017 '
    'sejam descartadas durante a conversão. A coluna original foi preservada e a coluna '
    '`date_parsed` foi criada apenas para a análise temporal.\n'
)

if len(miss):
    report.append(
        'As colunas originais com valores ausentes são:\n\n' +
        '\n'.join([
            f"- `{k}`: {int(v):,} ({v / len(df):.1%})"
            for k, v in miss.items()
        ]) + '\n'
    )
else:
    report.append('Não há valores ausentes nas colunas originais.\n')

if len(cat_counts):
    report.append('## Categorias\n')
    report.append(
        'As categorias mais frequentes são:\n\n' +
        '\n'.join([
            f"- `{k}`: {v / len(df):.1%}"
            for k, v in cat_counts.head(10).items()
        ]) + '\n'
    )

if 'label' in df.columns and key:
    medians = df.groupby('label')[key].median().round(3)
    medians.to_csv(os.path.join(OUT, 'medians_by_label.csv'))
    report.append('## Diferenças descritivas entre rótulos\n')
    report.append(
        'As medianas das variáveis numéricas por rótulo estão no arquivo '
        '`medians_by_label.csv`. Os boxplots foram construídos sem exibir outliers para '
        'facilitar a comparação visual. Esta EDA é descritiva e não demonstra causalidade '
        'nem desempenho preditivo fora da amostra.\n'
    )

report.append('## Arquivos gerados\n')
report.append(
    'A pasta inclui gráficos PNG, tabelas CSV, um resumo JSON e as estatísticas agrupadas.\n'
)

report.append('## Referências\n')
report.append(
    'Esta EDA foi calculada diretamente a partir do arquivo CSV fornecido pelo usuário; '
    'não foram usados dados externos.\n'
)

with open(os.path.join(OUT, 'EDA_Roney_Vinicius.md'), 'w', encoding='utf-8') as f:
    f.write('\n'.join(report))

# 13. Exportar todos os resultados em ZIP
zip_path = '/content/eda_roney_vinicius_resultados.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, dirs, filenames in os.walk(OUT):
        for filename in filenames:
            caminho = os.path.join(root, filename)
            z.write(caminho, arcname=os.path.relpath(caminho, '/content'))

print('\n' + '=' * 60)
print('EDA concluída com sucesso!')
print('=' * 60)
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('\nArquivos salvos em:', OUT)
print('Pacote ZIP:', zip_path)

# Baixar automaticamente o pacote final
files.download(zip_path)

# Para consultar os arquivos individualmente, use por exemplo:
# display(pd.read_csv('/content/eda_output/annual_counts.csv'))
# display(pd.read_csv('/content/eda_output/medians_by_label.csv'))
# display(pd.read_csv('/content/eda_output/missing_by_column.csv'))
# display(pd.read_csv('/content/eda_output/correlation_matrix.csv'))
# display(pd.read_csv('/content/eda_output/group_stats_by_label.csv'))
# display(pd.read_csv('/content/eda_output/group_stats_by_category.csv'))
# display(pd.read_csv('/content/eda_output/EDA_Roney_Vinicius.md', sep='\n', header=None))

# Se não quiser download automático, comente a linha:
# files.download(zip_path)
# e execute manualmente:
# files.download('/content/eda_roney_vinicius_resultados.zip')

print('\nMedianas por rótulo:')
if 'label' in df.columns and key:
    display(df.groupby('label')[key].median().round(3))

print('\nContagem por ano:')
if os.path.exists(os.path.join(OUT, 'annual_counts.csv')):
    display(pd.read_csv(os.path.join(OUT, 'annual_counts.csv')))


TypeError: 'NoneType' object is not subscriptable